*This code is a companion to the book **Mastering PyTorch and Lightning: A Step-by-Step Practical Guide with QA** by Aghiles Kebaili*

> This notebook contains the raw code for the Capstone Project. To understand the architectural decisions, the math behind the latent space, and the production *Gotchas* to avoid, get the full step-by-step guide on Amazon: **[Get the Book Here](https://www.amazon.fr/Mastering-PyTorch-Lightning-Step-Step-ebook/dp/B0HGNZS55V)**

## 1. Multi-Device Management with Device-Agnostic Code
### Step 1: Select and Verify the Execution Device

In [7]:
import torch
import torch.nn as nn

# Use torch.accelerator for device-agnostic code that works across CUDA, MPS, XPU, or CPU.
# This is more portable than hardcoding .cuda() or backend-specific conditionals.
if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
else:
    device = torch.device("cpu")

print(f"Executing on: {device}")

def build_model():
    return nn.Sequential(
        nn.Linear(4, 32),
        nn.ReLU(),
        nn.Linear(32, 3),
    )

model = build_model().to(device)

sample_inputs = torch.randn(8, 4)
sample_inputs = sample_inputs.to(device)
sample_outputs = model(sample_inputs)

assert next(model.parameters()).device.type == device.type
assert sample_inputs.device.type == device.type
assert sample_outputs.shape == (8, 3)

Executing on: cuda


### Step 2: Keep Custom Modules Device-Agnostic

In [ ]:
import torch
import torch.nn as nn

# Never hardcode devices inside a module. Temporary tensors should inherit device/dtype from inputs.
# Use functions like ones_like(), zeros_like() to match the input configuration.
class AgnosticLayer(nn.Module):
    def __init__(self, feature_size):
        super().__init__()
        self.weight = nn.Parameter(
            torch.randn(feature_size)
        )
        self.register_buffer(
            "center", torch.zeros(feature_size)
        )

    def forward(self, x):
        # Temporary tensors inherit the input configuration.
        mask = torch.ones_like(x)
        return (x - self.center) * self.weight * mask

layer = AgnosticLayer(feature_size=4).to(device)
layer_input = torch.randn(2, 4, device=device)
layer_output = layer(layer_input)

assert layer_output.device.type == device.type
assert layer.weight.requires_grad
assert "center" in dict(layer.named_buffers())
assert "center" in layer.state_dict()

## 2. Model Persistence: Saving & Loading
### Step 1: Separate Architecture from Learned State

In [10]:
state_dict = model.state_dict()

# state_dict() contains only registered parameters and buffers, not the model architecture.
# Save only state_dict, not the whole model, for portability and to support weights_only=True.
print(state_dict.keys())
assert "0.weight" in state_dict
assert "0.bias" in state_dict
assert "2.weight" in state_dict
assert "2.bias" in state_dict

odict_keys(['0.weight', '0.bias', '2.weight', '2.bias'])


### Step 2: Perform a Weight Round-Trip

In [ ]:
model.eval()
reference_input = torch.randn(4, 4, device=device)

# Always verify a weight round-trip: save, restore, and compare predictions.
# This test catches many persistence bugs early.
with torch.inference_mode():
    reference_output = model(reference_input).cpu()

torch.save(model.state_dict(), "weights_only.pth")

restored_model = build_model()

state_dict = torch.load(
    "weights_only.pth",
    map_location="cpu",
    weights_only=True,
)
restored_model.load_state_dict(state_dict)
restored_model.eval()

with torch.inference_mode():
    restored_output = restored_model(reference_input.cpu())

assert torch.allclose(reference_output, restored_output)

### Step 3: Save the Complete Training State

In [ ]:
import torch
import torch.optim as optim

# A full checkpoint must include model state, optimizer state, and any other stateful components.
# Adaptive optimizers maintain momentum and variance; restoring these is essential for resuming training.
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
completed_epoch = 4

# Populate the optimizer's historical state with one update.
model.train()
checkpoint_inputs = torch.randn(8, 4, device=device)
checkpoint_targets = torch.randint(0, 3, (8,), device=device)
optimizer.zero_grad(set_to_none=True)
checkpoint_loss = nn.functional.cross_entropy(
    model(checkpoint_inputs), checkpoint_targets
)
checkpoint_loss.backward()
optimizer.step()
current_loss = checkpoint_loss.item()

checkpoint = {
    "epoch": completed_epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": float(current_loss),
}
torch.save(checkpoint, "training_checkpoint.pth")

# Restoring the model and optimizer from the checkpoint.
checkpoint = torch.load(
    "training_checkpoint.pth",
    map_location="cpu",
    weights_only=True,
)

resumed_model = build_model().to(device)
resumed_model.load_state_dict(
    checkpoint["model_state_dict"]
)

resumed_optimizer = optim.AdamW(
    resumed_model.parameters(), lr=1e-3
)
resumed_optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

start_epoch = checkpoint["epoch"] + 1

assert start_epoch == 5
assert isinstance(checkpoint["loss"], float)
assert len(resumed_optimizer.state) > 0

## 3. Gotchas & Reality Checks: Production Pitfalls
### Gotcha 1: `Tensor.to()` vs `Module.to()`

In [16]:
model.to(device)

# Wrong when a transfer is required: return value is discarded.
original_inputs = torch.randn(2, 4)
original_inputs.to(device)

# Correct: retain the returned tensor.
inputs = original_inputs.to(device)

assert inputs.device.type == device.type
if device.type != "cpu":
    assert original_inputs.device.type == "cpu"

### Gotcha 2: Checkpointing a Compiled Wrapper

In [19]:
base_model = build_model().to(device)
compiled_model = torch.compile(base_model)

# Train through compiled_model, but persist the stable module state.
torch.save(base_model.state_dict(), "model_state.pth")

### Gotcha 3: Restoring Accelerator Tensors on a CPU Host

In [23]:
checkpoint = torch.load("training_checkpoint.pth", map_location="cpu")